## Phase 2 — LoRA Adapter Training

**IMPORTANT: Restart kernel before running this phase.**
vLLM and Unsloth cannot share a Python session.

Four adapters are trained independently:
- B1: θ_inst + D1 (instruction, same-manifold)
- B2: θ_inst + D2 (math, same-manifold)  
- C1: θ_base + D1 (instruction, cross-manifold)
- C2: θ_base + D2 (math, cross-manifold)

Config: r=16 · α=16 · dropout=0.0 · 7 modules · 1 epoch · lr=2e-4

In [ ]:
# Phase 2 uses Unsloth only
# Do NOT import vLLM in this session

import subprocess
subprocess.run([
    "pip", "install",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "--quiet"
])
subprocess.run([
    "pip", "install",
    "peft", "transformers", "datasets",
    "trl", "accelerate", "bitsandbytes",
    "--quiet"
])

print("Unsloth installed.")
print("Do not import vLLM in this session.")

In [ ]:
import os
import gc
import json
import torch
import random
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

random.seed(42)
torch.manual_seed(42)

# ── Model config ──────────────────────────────────────────
STUDENT_INSTRUCT = "Qwen/Qwen2.5-7B-Instruct"
STUDENT_BASE     = "Qwen/Qwen2.5-7B"
MAX_SEQ_LENGTH   = 2048

# ── LoRA config ───────────────────────────────────────────
LORA_RANK        = 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0
TARGET_MODULES   = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# ── Training config ───────────────────────────────────────
TRAIN_SAMPLES    = 5000
EPOCHS           = 1
LEARNING_RATE    = 2e-4
BATCH_SIZE       = 2
GRAD_ACCUM       = 4
WARMUP_STEPS     = 10

# ── Paths ─────────────────────────────────────────────────
D1_PATH          = "/kaggle/working/d1_instruction.jsonl"
D2_PATH          = "/kaggle/working/d2_math.jsonl"
OUTPUT_BASE      = "/kaggle/working"

print("Config loaded.")
print(f"Target modules: {TARGET_MODULES}")
print(f"LoRA: r={LORA_RANK}, α={LORA_ALPHA}, dropout={LORA_DROPOUT}")

In [ ]:
CHAT_TEMPLATE = (
    "<|im_start|>user\n{instruction}<|im_end|>\n"
    "<|im_start|>assistant\n{response}<|im_end|>"
)

def load_distillation_dataset(path, n=5000):
    """Load JSONL distillation dataset and format with chat template."""
    with open(path, "r") as f:
        lines = f.readlines()

    entries = [json.loads(l) for l in lines[:n]]

    texts = [
        CHAT_TEMPLATE.format(
            instruction=e["instruction"],
            response=e["response"]
        )
        for e in entries
    ]

    dataset = Dataset.from_dict({"text": texts})
    print(f"Loaded {len(dataset)} samples from {path}")
    print(f"Example:\n{texts[0][:200]}")
    return dataset

print("Dataset loader ready.")

In [ ]:
def train_lora(
    base_model_name,
    dataset,
    output_path,
    adapter_name,
):
    """
    Train a single LoRA adapter.
    
    Args:
        base_model_name: θ_inst or θ_base
        dataset:         distillation dataset
        output_path:     where to save adapter
        adapter_name:    B1, B2, C1, or C2
    """
    print(f"\n{'='*60}")
    print(f"Training {adapter_name}")
    print(f"Base: {base_model_name}")
    print(f"Samples: {len(dataset)}")
    print(f"{'='*60}")

    # Load base model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = base_model_name,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = False,
    )

    # Inject LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r                   = LORA_RANK,
        lora_alpha          = LORA_ALPHA,
        lora_dropout        = LORA_DROPOUT,
        target_modules      = TARGET_MODULES,
        bias                = "none",
        use_gradient_checkpointing = "unsloth",
        random_state        = 42,
    )

    print(f"LoRA injected: r={LORA_RANK}, α={LORA_ALPHA}")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Training arguments
    training_args = TrainingArguments(
        output_dir              = output_path,
        num_train_epochs        = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate           = LEARNING_RATE,
        warmup_steps            = WARMUP_STEPS,
        logging_steps           = 50,
        save_strategy           = "no",
        fp16                    = not torch.cuda.is_bf16_supported(),
        bf16                    = torch.cuda.is_bf16_supported(),
        seed                    = 42,
        report_to               = "none",
    )

    # Trainer
    trainer = SFTTrainer(
        model        = model,
        tokenizer    = tokenizer,
        train_dataset= dataset,
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LENGTH,
        args               = training_args,
    )

    # Train
    trainer.train()

    # Save adapter only — not full model
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)

    adapter_size = sum(
        os.path.getsize(os.path.join(output_path, f))
        for f in os.listdir(output_path)
    ) / 1e6

    print(f"\n{adapter_name} saved to {output_path}")
    print(f"Adapter size: {adapter_size:.1f} MB")

    # Free VRAM
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    print(f"VRAM freed after {adapter_name}")
    return output_path

In [ ]:
# ── Adapter B1 ────────────────────────────────────────────
# Same-manifold · Instruction domain
# Reference: θ_inst

d1_dataset = load_distillation_dataset(D1_PATH, n=TRAIN_SAMPLES)

train_lora(
    base_model_name = STUDENT_INSTRUCT,
    dataset         = d1_dataset,
    output_path     = f"{OUTPUT_BASE}/lora_B1",
    adapter_name    = "B1",
)

print("B1 complete.")

In [ ]:
# ── Adapter B2 ────────────────────────────────────────────
# Same-manifold · Math domain
# Reference: θ_inst

d2_dataset = load_distillation_dataset(D2_PATH, n=TRAIN_SAMPLES)

train_lora(
    base_model_name = STUDENT_INSTRUCT,
    dataset         = d2_dataset,
    output_path     = f"{OUTPUT_BASE}/lora_B2",
    adapter_name    = "B2",
)

print("B2 complete.")

In [ ]:
# ── Adapter C1 ────────────────────────────────────────────
# Cross-manifold · Instruction domain
# Reference: θ_base

train_lora(
    base_model_name = STUDENT_BASE,
    dataset         = d1_dataset,
    output_path     = f"{OUTPUT_BASE}/lora_C1",
    adapter_name    = "C1",
)

print("C1 complete.")

In [ ]:
# ── Adapter C2 ────────────────────────────────────────────
# Cross-manifold · Math domain
# Reference: θ_base

train_lora(
    base_model_name = STUDENT_BASE,
    dataset         = d2_dataset,
    output_path     = f"{OUTPUT_BASE}/lora_C2",
    adapter_name    = "C2",
)

print("C2 complete.")

In [ ]:
adapters = ["lora_B1", "lora_B2", "lora_C1", "lora_C2"]

print("Adapter verification:")
print(f"{'Adapter':<12} {'Path':<40} {'Size (MB)':<12} {'Status'}")
print("-" * 70)

for adapter in adapters:
    path = f"{OUTPUT_BASE}/{adapter}"
    if os.path.exists(path):
        size = sum(
            os.path.getsize(os.path.join(path, f))
            for f in os.listdir(path)
        ) / 1e6
        status = "✓"
    else:
        size   = 0
        status = "✗ MISSING"

    print(f"{adapter:<12} {path:<40} {size:<12.1f} {status}")

print("\nPhase 2 complete.")
print("Restart kernel before running Phase 3.")